# Brainclip Remotion Render Engine (Colab)
This notebook sets up a high-performance rendering API on Google Colab to offload video generation from AWS Lambda.

In [ ]:
!echo "Installing Node.js..."
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
!apt-get install -y nodejs

!echo "Installing Remotion CLI globally..."
!npm install -g remotion@4.0.438

!echo "Installing Headless Chrome dependencies..."
!apt-get install -y \
    libnss3 libxss1 libasound2 libatk-bridge2.0-0 \
    libgtk-3-0 libgbm1 libxshmfence1 xvfb

!echo "Installing Python API dependencies..."
!pip install fastapi uvicorn nest-asyncio pyngrok requests

In [ ]:
import os
import json
import subprocess
import requests
import nest_asyncio
import uuid
from pyngrok import ngrok
from fastapi import FastAPI, HTTPException, BackgroundTasks
from pydantic import BaseModel
from typing import Any, Dict

app = FastAPI()
tasks = {}

class RenderRequest(BaseModel):
    serveUrl: str
    composition: str = "ReelComposition"
    inputProps: Dict[str, Any]
    uploadUrl: str

def do_render(req: RenderRequest, job_id: str):
    print(f"[{job_id}] Starting render task...")
    props_path = f"props_{job_id}.json"
    out_path = f"out_{job_id}.mp4"
    
    with open(props_path, "w") as f:
        json.dump(req.inputProps, f)
        
    cmd = [
        "npx", "remotion", "render",
        req.serveUrl,
        req.composition,
        out_path,
        f"--props={props_path}",
        "--concurrency=max"
    ]
    
    try:
        subprocess.run(cmd, capture_output=True, text=True, check=True)
        print(f"[{job_id}] Render complete, uploading...")
        
        with open(out_path, "rb") as f:
            headers = {"Content-Type": "video/mp4"}
            upload_res = requests.put(req.uploadUrl, data=f, headers=headers)
            
        if upload_res.status_code not in (200, 201):
            tasks[job_id] = {"status": "failed", "error": f"Upload failed: {upload_res.text}"}
        else:
            tasks[job_id] = {"status": "done"}
            print(f"[{job_id}] Upload successful!")
            
    except subprocess.CalledProcessError as e:
        print(f"[{job_id}] Render failed!")
        print(e.stderr)
        tasks[job_id] = {"status": "failed", "error": "Render failed."}
    except Exception as e:
        tasks[job_id] = {"status": "failed", "error": str(e)}
    finally:
        if os.path.exists(out_path):
            os.remove(out_path)
        if os.path.exists(props_path):
            os.remove(props_path)

@app.post("/render")
async def render_video(req: RenderRequest, background_tasks: BackgroundTasks):
    job_id = str(uuid.uuid4())
    tasks[job_id] = {"status": "rendering"}
    background_tasks.add_task(do_render, req, job_id)
    return {"job_id": job_id}

@app.get("/render/{job_id}")
async def get_status(job_id: str):
    return tasks.get(job_id, {"status": "not_found"})

# --- Start Server ---
# ngrok.set_auth_token("YOUR_NGROK_AUTH_TOKEN")

ngrok_tunnel = ngrok.connect(8000)
print("==================================================")
print(f"🚀 Colab Render API URL: {ngrok_tunnel.public_url}")
print("==================================================")

nest_asyncio.apply()
import uvicorn
uvicorn.run(app, host="0.0.0.0", port=8000)
